## Pré-processamento dos dados do dataset WineQT

### Parte 2.1 - Carga dos dados e criação da coluna target

In [20]:
# Importa as bibliotecas necessárias
import sys
import os

# Pega o diretório atual exato onde o notebook está rodando
diretorio_atual = os.getcwd()

# Descobre a raiz do projeto
if diretorio_atual.endswith('notebooks'):
    caminho_raiz = os.path.abspath(os.path.join(diretorio_atual, '..'))
else:
    # Caso o VS Code já esteja rodando a partir da raiz
    caminho_raiz = diretorio_atual

# Adiciona a raiz aos caminhos do Python
if caminho_raiz not in sys.path:
    sys.path.append(caminho_raiz)

print(f'Caminho raiz configurado para: {caminho_raiz}.')

# Importa a função de carga do dataset
from src.load import carregar_dataset
print('\nFunção de carregamento importada com sucesso.')

# Carrega o dataset e cria a coluna alvo
df = carregar_dataset('../data/WineQT.csv', 'quality')
print('\nDataset carregado e coluna alvo criada com sucesso.')
print(f'\nO dataset possui {df.shape[0]} linhas e {df.shape[1]} colunas.')

Caminho raiz configurado para: c:\z_FIAP\wine-quality-classification.

Função de carregamento importada com sucesso.

Dataset carregado e coluna alvo criada com sucesso.

O dataset possui 1143 linhas e 14 colunas.


### Parte 2.2 - Tratamento dos Dados Faltantes

A Análise Exploratória (EDA) prévia confirmou a ausência de valores nulos no dataset atual. No entanto, para garantir a resiliência do *pipeline* de dados em produção, está sendo implementada a remoção de eventuais registros com falhas de medição.

In [21]:
## Remoção dos registros com valores nulos

# Verifica o tamanho original
tamanho_original = len(df)

# Remove linhas com valores nulos (Dropna)
df = df.dropna()

linhas_removidas = tamanho_original - len(df)

if linhas_removidas > 0:
    print(f"{linhas_removidas} linhas com dados faltantes foram removidas.")
else:
    print("Não foram encontradas linhas com dados faltantes.")

Não foram encontradas linhas com dados faltantes.


### Parte 2.3 - Criação de Novas Features por meio de Feature Engineering

Para ajudar os modelos de ML a capturararem padrões mais complexos, criaremos novas propriedades baseadas no conhecimento de vinhos:
1. **Acidez total:** É a soma da acidez fixa com a acidez volátil. O equilíbrio da acidez é muito importante para a qualidade do vinho.
2. **Razão açúcar/álcool:** Vinhos com alta taxa de conversão de açúcar em álcool costumam ter perfis de sabor diferentes, sendo mais secos.

In [22]:
# Criação da feature Acidez Total
df['total_acidity'] = df['fixed acidity'] + df['volatile acidity']

# Criação da feature Razão Açúcar/Álcool
# Observação: A adição da constante matemática (1e-5) evita erro de divisão por zero
df['sugar_to_alcohol_ratio'] = df['residual sugar'] / (df['alcohol'] + 1e-5)

print('\nNovas features criadas com sucesso!')
print('\nVisualização prévia das novas features:')
print(df[['total_acidity', 'sugar_to_alcohol_ratio', 'target']].head())


Novas features criadas com sucesso!

Visualização prévia das novas features:
   total_acidity  sugar_to_alcohol_ratio  target
0           8.10                0.202127       0
1           8.68                0.265306       0
2           8.56                0.234694       0
3          11.48                0.193877       0
4           8.10                0.202127       0


### Parte 2.4 - Divisão dos Dados entre Treino e Teste

Antes de aplicar qualquer transformação dos dados, é necessário dividir o dataset entre treino e teste. Isso evita o **Data Leakage**, garantindo que o modelo não tenha acesso prévio à média e variância do conjunto de validação.

Devido ao desbalanceamento das classes, usaremos a separação estratificada.

In [23]:
# Importa a função de divisão dos dados em treino e teste do módulo src
from src.data_transformation import dividir_dados_treino_teste

# Define a proporção de divisão entre treino e teste
proporcao_teste = 0.2
proporcao_treino = 1 - proporcao_teste

# Chama a função modularizada
x_train, x_test, y_train, y_test = dividir_dados_treino_teste(df, ['Id', 'quality', 'target'], 'target', proporcao_teste, 42)

print(f'Divisão entre treino e teste concluída com sucesso, na proporção {int(proporcao_treino * 100)}/{int(proporcao_teste * 100)}.')
print(f'- Conjunto de treino: {x_train.shape[0]} amostras.')
print(f'- Conjunto de teste: {x_test.shape[0]} amostras.')

Divisão entre treino e teste concluída com sucesso, na proporção 80/20.
- Conjunto de treino: 914 amostras.
- Conjunto de teste: 229 amostras.


### Parte 2.5 - Padronização dos Dados

Como constatado na etapa de EDA, as grandezas químicas estão em escalas muito diferentes. Dessa forma, aplicaremos o `StandardScaler` para garantir que variáveis com valores absolutos elevados não dominem matematicamente o algoritmo.

In [24]:
# Importa a função de padronização do módulo src
from src.data_transformation import aplicar_padronizacao

# Define onde o arquivo do Scaler será salvo
caminho_scaler = os.path.join(caminho_raiz, 'results', 'scaler_vinho.pkl')

# Chama a função modularizada
x_train_scaled, x_test_scaled = aplicar_padronizacao(x_train, x_test, caminho_scaler)

print(f'Dados padronizados com sucesso e Scaler salvo em {caminho_scaler}.')

# Exibe a prévia do resultado da padronização para validação
x_train_scaled.head()

Dados padronizados com sucesso e Scaler salvo em c:\z_FIAP\wine-quality-classification\results\scaler_vinho.pkl.


,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,total_acidity,sugar_to_alcohol_ratio
0,-1.320971,-0.165638,-1.400862,-0.804309,-0.677051,-0.024763,-0.580648,-1.162557,0.317862,-1.240228,-0.868305,-1.362632,-0.676647
1,1.124820,-0.681264,0.772796,-0.107462,3.055419,-0.928089,-0.942127,1.360502,-0.777490,-0.217749,-0.868305,1.077117,0.042109
2,0.612910,-1.082306,0.514027,-0.246832,-0.246381,-1.028459,-0.972250,0.642558,-0.584193,-0.217749,-1.144489,0.514990,-0.050079
3,-0.126515,4.589577,-1.400862,-0.525570,-0.078898,-1.229198,-1.002373,-0.177949,1.477647,-1.059790,0.420555,0.335341,-0.555316
4,-0.638425,0.006237,-0.728063,-0.386201,-0.653125,0.276346,-0.701141,-0.521536,-0.648625,0.143126,-0.500059,-0.649831,-0.303475
